# Exploring the RevisitOP benchmark

Loads `roxford5k` and `rparis6k` via `cbir.data.revisitop.download`, which wraps the
`galilai-group/revisitop` HF dataset script (`trust_remote_code=True`, requires
`datasets<4.0`) and asserts the loaded splits match the verified counts in
`AGENTS.md` before returning anything.

Note: the `revisitop1m` and `oxfordparis` configs in that loading script are known to
be broken (see AGENTS.md) — `download()` rejects them outright, so they are
intentionally not used here.

You can also fetch these ahead of time from the shell: `uv run cbir download --datasets roxford5k rparis6k`.

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt
from PIL import ImageDraw

from cbir.data.revisitop import download

## Load roxford5k

In [ ]:
ox_query, ox_db = download("roxford5k")
print(ox_query)
print(ox_db)

In [ ]:
ox_query.features


In [ ]:
example = ox_query[0]
{k: v for k, v in example.items() if k != "image"}

In [ ]:
# Draw the query bbox on the full image
img = example["image"].convert("RGB").copy()
draw = ImageDraw.Draw(img)
x1, y1, x2, y2 = example["bbx"]
draw.rectangle([x1, y1, x2, y2], outline="red", width=4)

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(img)
title = f"{example['filename']} (easy={len(example['easy'])}, hard={len(example['hard'])}, junk={len(example['junk'])})"
ax.set_title(title)
ax.axis("off")

In [ ]:
# Distribution of #easy / #hard / #junk per query, and how many queries have zero easy positives
easy_counts = [len(q["easy"]) for q in ox_query]
hard_counts = [len(q["hard"]) for q in ox_query]
junk_counts = [len(q["junk"]) for q in ox_query]

print("queries with 0 easy positives:", sum(1 for c in easy_counts if c == 0))
print("queries with 0 hard positives:", sum(1 for c in hard_counts if c == 0))

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist([easy_counts, hard_counts, junk_counts], bins=15, label=["easy", "hard", "junk"])
ax.set_xlabel("# relevant images per query")
ax.set_ylabel("# queries")
ax.set_title("roxford5k: relevance-set sizes per query")
ax.legend()

In [ ]:
Counter(ox_db["dataset"])


In [ ]:
# One of query 0's "easy" ground-truth matches, retrieved from the database by index
easy_id = example["easy"][10]
easy = ox_db[easy_id]
img = easy["image"]
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(img)
ax.set_title(easy["filename"])
ax.axis("off")

## Load rparis6k

In [ ]:
pa_query, pa_db = download("rparis6k")

In [ ]:
pa_example = pa_query[0]
img = pa_example["image"].convert("RGB").copy()
draw = ImageDraw.Draw(img)
x1, y1, x2, y2 = pa_example["bbx"]
draw.rectangle([x1, y1, x2, y2], outline="red", width=4)

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(img)
ax.set_title(f"{pa_example['filename']}  (easy={len(pa_example['easy'])}, hard={len(pa_example['hard'])}, junk={len(pa_example['junk'])})")
ax.axis("off")

In [ ]:
easy_counts_p = [len(q["easy"]) for q in pa_query]
hard_counts_p = [len(q["hard"]) for q in pa_query]
junk_counts_p = [len(q["junk"]) for q in pa_query]

print("queries with 0 easy positives:", sum(1 for c in easy_counts_p if c == 0))
print("queries with 0 hard positives:", sum(1 for c in hard_counts_p if c == 0))

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist([easy_counts_p, hard_counts_p, junk_counts_p], bins=15, label=["easy", "hard", "junk"])
ax.set_xlabel("# relevant images per query")
ax.set_ylabel("# queries")
ax.set_title("rparis6k: relevance-set sizes per query")
ax.legend()